# WasteWise: Exploratory Data Analysis

Exploration of the Food Demand Forecasting dataset (weekly orders per fulfilment center and meal).
Each section ends with the findings that feed into cleaning (Phase 3) and feature engineering (Phase 4).

Input: `data/interim/train_merged.parquet`, produced by `make interim`.
Column meanings are in `docs/data_dictionary.md`.

In [1]:
import pandas as pd

from wastewise.config import INTERIM_DATA_DIR, MERGED_TRAIN_FILE
from wastewise.data.load import load_interim_merged

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

df = load_interim_merged()
print(f"Loaded {INTERIM_DATA_DIR / MERGED_TRAIN_FILE}")

Loaded C:\Users\Pongo\Documents\Coding\Antigravity\WasteWise\data\interim\train_merged.parquet


## 1. Dataset overview

Size, time span, and how many centers, meals and center-meal series the data covers.

In [2]:
print(f"Rows: {len(df):,}  Columns: {df.shape[1]}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head()

Rows: 456,548  Columns: 15
Memory: 23.3 MB


,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,city_code,region_code,center_type,op_area,category,cuisine
0,1379560,1,55,1885,136.83,152.29,0,0,177,647,56,TYPE_C,2.0,Beverages,Thai
1,1466964,1,55,1993,136.83,135.83,0,0,270,647,56,TYPE_C,2.0,Beverages,Thai
2,1346989,1,55,2539,134.86,135.86,0,0,189,647,56,TYPE_C,2.0,Beverages,Thai
3,1338232,1,55,2139,339.50,437.53,0,0,54,647,56,TYPE_C,2.0,Beverages,Indian
4,1448490,1,55,2631,243.50,242.50,0,0,40,647,56,TYPE_C,2.0,Beverages,Indian


In [3]:
df.dtypes.to_frame("dtype").T

,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,city_code,region_code,center_type,op_area,category,cuisine
dtype,int64,int16,int16,int16,float64,float64,int8,int8,int32,int16,int16,category,float64,category,category


In [4]:
n_weeks = df["week"].nunique()
n_centers = df["center_id"].nunique()
n_meals = df["meal_id"].nunique()
n_series = df.groupby(["center_id", "meal_id"], observed=True).ngroups

overview = pd.Series(
    {
        "first week": df["week"].min(),
        "last week": df["week"].max(),
        "distinct weeks": n_weeks,
        "centers": n_centers,
        "meals": n_meals,
        "center-meal series observed": n_series,
        "center-meal pairs possible": n_centers * n_meals,
        "series coverage (%)": round(100 * n_series / (n_centers * n_meals), 1),
        "rows if every series had every week": n_series * n_weeks,
        "row fill rate (%)": round(100 * len(df) / (n_series * n_weeks), 1),
    },
    dtype="object",
)
overview.to_frame("value")

,value
first week,1
last week,145
distinct weeks,145
centers,77
meals,51
center-meal series observed,3597
center-meal pairs possible,3927
series coverage (%),91.6
rows if every series had every week,521565
row fill rate (%),87.5


In [5]:
dims = {
    "city_code": df["city_code"].nunique(),
    "region_code": df["region_code"].nunique(),
    "center_type": df["center_type"].nunique(),
    "category": df["category"].nunique(),
    "cuisine": df["cuisine"].nunique(),
}
pd.Series(dims, name="distinct values").to_frame()

,distinct values
city_code,51
region_code,8
center_type,3
category,14
cuisine,4


In [6]:
rows_per_week = df.groupby("week").size()
rows_per_week.describe().round(1).to_frame("rows per week").T

,count,mean,std,min,25%,50%,75%,max
rows per week,145.0,3148.6,150.7,2786.0,3044.0,3205.0,3263.0,3359.0


In [7]:
weeks_per_series = df.groupby(["center_id", "meal_id"], observed=True)["week"].nunique()
weeks_per_series.describe().round(1).to_frame("weeks per series").T

,count,mean,std,min,25%,50%,75%,max
weeks per series,3597.0,126.9,28.8,1.0,122.0,140.0,145.0,145.0


In [8]:
df[["checkout_price", "base_price", "num_orders", "op_area"]].describe().round(2)

,checkout_price,base_price,num_orders,op_area
count,456548.00,456548.00,456548.00,456548.00
mean,332.24,354.16,261.87,4.08
std,152.94,160.72,395.92,1.09
min,2.97,55.35,13.00,0.90
25%,228.95,243.50,54.00,3.60
50%,296.82,310.46,136.00,4.00
75%,445.23,458.87,324.00,4.50
max,866.27,866.27,24299.00,7.00


### Findings: dataset overview

- **Size**: 456,548 rows x 15 columns (23.3 MB in memory) after merging center and meal info.
- **Time span**: 145 consecutive weeks (week 1 to 145), roughly 2.8 years of weekly data.
  There are no calendar dates, only the week index.
- **Coverage**: 77 centers x 51 meals = 3,927 possible pairs, of which 3,597 (91.6%) appear.
  Not every center sells every meal.
- **Gaps**: a full panel would have 521,565 rows (3,597 series x 145 weeks); the data has 87.5% of that.
  Rows per week range from 2,786 to 3,359, so the set of active series changes over time.
- **Short series**: weeks per series has a median of 140 but a minimum of 1 and a 25th percentile of 122.
  Very short series will have few or no lag features; completeness is analysed in section 2.8.
- **Target scale**: `num_orders` has mean 261.87 vs median 136 and a max of 24,299, a strong right skew
  (section 2.3 checks whether a log transform is needed).
- **Hierarchy available for aggregation**: 51 cities, 8 regions, 3 center types; 14 meal categories,
  4 cuisines.